# SHIPIT Agent: Streaming Events And Packets

Use this notebook to inspect the real-time runtime stream, collected event packets, and frontend-ready packet transports.

In [4]:
from pathlib import Path
import os
import sys
import json
import textwrap
from IPython.display import Markdown, display

# --- Load credentials from .env ------------------------------------------
# `build_llm_from_env` also auto-loads a .env by walking upward from CWD,
# but we load it here too so the notebook can print a clear status line
# BEFORE building the LLM. Works with either python-dotenv (if installed)
# or the stdlib loader shipped with examples/run_multi_tool_agent.py.
try:
    from dotenv import load_dotenv  # optional dep — only used if present
    env_path = Path.cwd().parent / '.env' if Path.cwd().name == 'notebooks' else Path.cwd() / '.env'
    load_dotenv(dotenv_path=env_path, override=False)
except ImportError:
    pass  # Stdlib loader inside build_llm_from_env will handle it

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from examples.run_multi_tool_agent import build_demo_agent, build_llm_from_env, load_env_file

# Double-sure .env is loaded via shipit's own stdlib loader too.
loaded = load_env_file()

# Print a visibility block so you can see which provider will be used and
# which credentials are actually available BEFORE the LLM is constructed.
print(f"Loaded {len(loaded)} vars via shipit .env loader" if loaded else "No .env picked up by shipit loader")
print(f"SHIPIT_LLM_PROVIDER = {os.getenv('SHIPIT_LLM_PROVIDER', '(unset → defaults to bedrock)')}")
print(f"SHIPIT_OPENAI_MODEL = {os.getenv('SHIPIT_OPENAI_MODEL', '(unset → defaults to gpt-4o-mini)')}")
for key in ('OPENAI_API_KEY', 'ANTHROPIC_API_KEY', 'AWS_REGION_NAME', 'GOOGLE_API_KEY'):
    print(f"  {key}: {'✓ set' if os.getenv(key) else '✗ missing'}")

No .env picked up by shipit loader
SHIPIT_LLM_PROVIDER = (unset → defaults to bedrock)
SHIPIT_OPENAI_MODEL = gpt-4o-mini
  OPENAI_API_KEY: ✓ set
  ANTHROPIC_API_KEY: ✗ missing
  AWS_REGION_NAME: ✗ missing
  GOOGLE_API_KEY: ✗ missing


In [5]:
def _short(val, n=220):
    s = val if isinstance(val, str) else json.dumps(val, default=str)
    return textwrap.shorten(s.replace('\n', ' '), width=n, placeholder=' …')

def pretty_event(ev):
    t, msg, p = ev.type, ev.message or '', ev.payload or {}
    if t == 'run_started':
        return f'[status] run started :: {_short(p.get("prompt", ""), 140)}'
    if t == 'planning_started':
        return f'[thinking] planning started :: {_short(p.get("prompt", ""), 140)}'
    if t == 'planning_completed':
        return f'[thinking] planning completed :: {_short(p.get("output", ""))}'
    if t == 'reasoning_started':
        return f'[thinking] 🧠 model reasoning started (iteration {p.get("iteration", "?")})'
    if t == 'reasoning_completed':
        return f'[thinking] 🧠 reasoning :: {_short(p.get("content", ""), 320)}'
    if t == 'step_started':
        return f'[step] LLM completion started :: iteration={p.get("iteration", "?")}, tools={p.get("tool_count", "?")}'
    if t == 'tool_called':
        return f'[tool] ▶ calling `{msg}` :: args={_short(p.get("arguments", {}))}'
    if t == 'tool_completed':
        return f'[tool] ✔ completed `{msg}` :: {_short(p.get("output", ""))}'
    if t == 'tool_failed':
        return f'[tool] ✖ failed `{msg}` :: {_short(p.get("error", ""))}'
    if t == 'interactive_request':
        return f'[interactive] {msg} :: {_short(p)}'
    if t == 'run_completed':
        return f'[final] {_short(p.get("output", ""))}'
    return f'[{t}] {msg} :: {_short(p)}'


In [6]:
llm = build_llm_from_env('openai')

agent = build_demo_agent(
    llm=llm,
    workspace_root=str(ROOT / '.shipit_notebooks' / 'streaming_demo'),
)

# A stronger prompt that forces the agent to:
#   1. Think out loud (reasoning step)
#   2. Call web_search (tool step)
#   3. Open at least one source (second tool step)
#   4. Compare findings and produce a final grounded answer
prompt = (
    "You are an autonomous research agent. Task: find today's Bitcoin (BTC) "
    "price in USD from TWO independent live sources, then compare them.\n\n"
    "Rules — you MUST follow every rule:\n"
    "1. First, think step-by-step about your plan before acting.\n"
    "2. Call `web_search` at least once to locate candidate sources.\n"
    "3. Call `open_url` on at least TWO different URLs to fetch actual prices.\n"
    "4. Do NOT answer from prior knowledge — only use what the tools return.\n"
    "5. Finish with a short markdown report: a table of (source, price, timestamp) "
    "   followed by a 1-sentence reconciliation.\n"
    "If tools fail, retry with a different query or URL before giving up."
)

In [7]:
# Live streaming status view with incremental notebook updates.
# Requires the runtime fix that makes AgentRuntime.stream() yield events as they are emitted.
from IPython.display import Markdown, clear_output, display

lines: list[str] = []
print('Waiting for first event...', flush=True)

for index, event in enumerate(agent.stream(prompt), start=1):
    lines.append(f'{index}. `{event.type}` — {pretty_event(event)}')
    clear_output(wait=True)
    display(Markdown('## Live Stream\n\n' + '\n'.join(lines)))

print(f'\nStream complete — {len(lines)} events received.')
lines

## Live Stream

1. `run_started` — [status] run started :: You are an autonomous research agent. Task: find today's Bitcoin (BTC) price in USD from TWO independent live sources, then compare them. …
2. `planning_started` — [thinking] planning started :: You are an autonomous research agent. Task: find today's Bitcoin (BTC) price in USD from TWO independent live sources, then compare them. …
3. `planning_completed` — [thinking] planning completed :: Goal: You are an autonomous research agent. Task: find today's Bitcoin (BTC) price in USD from TWO independent live sources, then compare them. Rules — you MUST follow every rule: 1. First, think step-by-step about …
4. `step_started` — [step] LLM completion started :: iteration=1, tools=28
5. `llm_retry` — [llm_retry] Retrying LLM completion :: {"attempt": 1, "error": "Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk- …
6. `llm_retry` — [llm_retry] Retrying LLM completion :: {"attempt": 2, "error": "Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk- …

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: sk-proj-********************************************************************************************************************************************************VLYA. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'code': 'invalid_api_key', 'param': None}, 'status': 401}

In [5]:
# Collected event packets in one object.
events = [event.to_dict() for event in agent.stream(prompt)]
events

open_url: Playwright fetch failed for https://finance.yahoo.com/quote/BTC-USD: BrowserType.launch: Executable doesn't exist at /Users/rahulraj/Library/Caches/ms-playwright/chromium_headless_shell-1155/chrome-mac/headless_shell
╔════════════════════════════════════════════════════════════╗
║ Looks like Playwright was just installed or updated.       ║
║ Please run the following command to download new browsers: ║
║                                                            ║
║     playwright install                                     ║
║                                                            ║
║ <3 Playwright Team                                         ║
╚════════════════════════════════════════════════════════════╝
open_url: Playwright fetch failed for https://api.coingecko.com/api/v3/simple/price?ids=bitcoin&vs_currencies=usd: BrowserType.launch: Executable doesn't exist at /Users/rahulraj/Library/Caches/ms-playwright/chromium_headless_shell-1155/chrome-mac/headless_shell
╔═════

[{'type': 'run_started',
  'message': 'Agent run started',
  'payload': {'prompt': "You are an autonomous research agent. Task: find today's Bitcoin (BTC) price in USD from TWO independent live sources, then compare them.\n\nRules — you MUST follow every rule:\n1. First, think step-by-step about your plan before acting.\n2. Call `web_search` at least once to locate candidate sources.\n3. Call `open_url` on at least TWO different URLs to fetch actual prices.\n4. Do NOT answer from prior knowledge — only use what the tools return.\n5. Finish with a short markdown report: a table of (source, price, timestamp)    followed by a 1-sentence reconciliation.\nIf tools fail, retry with a different query or URL before giving up."}},
 {'type': 'planning_started',
  'message': 'Planner started',
  'payload': {'prompt': "You are an autonomous research agent. Task: find today's Bitcoin (BTC) price in USD from TWO independent live sources, then compare them.\n\nRules — you MUST follow every rule:\n1. 

ModuleNotFoundError: No module named 'examples.run_multi_tool_agent'

In [ ]:
# Final output plus stored events from a non-streaming run.
result = agent.run(prompt)
print(result.output)
result.events[:5]

In [ ]:
session = agent.chat_session(session_id='stream-demo')

for packet in session.stream_packets(
    'Use tools if useful and return a compact answer.',
    transport='websocket',
):
    print(packet)

In [ ]:
session = agent.chat_session(session_id='stream-demo-sse')

for packet in session.stream_packets(
    'Use tools if useful and return a compact answer.',
    transport='sse',
):
    print(packet)